# Gaussian random fields - initial conditions

*Walkthrough notebook for `neural_operators/data/grf.py`.*

To train an operator we need many different **input functions**. The paper draws the initial vorticity from
a Gaussian random field with covariance

$$a\sim\mathcal N\!\Big(0,\;\tau^{2\alpha-d}\,(-\Delta+\tau^2 I)^{-\alpha}\Big),\qquad d=2,\ \alpha=2.5,\ \tau=7 .$$

On a periodic domain $-\Delta$ is diagonal in the Fourier basis, so we can sample cheaply:
draw white noise in Fourier space, multiply each coefficient by $\sqrt{\text{eigenvalue}}$, and inverse-FFT.
Larger $\alpha$ or $\tau$ gives smoother fields.

## Setup

The first cell makes the repository importable when this notebook is opened from `notebooks/source/`; the second holds the module's imports.

In [ ]:
# Make the repo root importable so `import neural_operators` works from notebooks/source/.
import sys, pathlib
ROOT = pathlib.Path.cwd().resolve()
while not (ROOT / "neural_operators").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

In [ ]:
import math
import torch

## The sampler

- `__init__` precomputes `sqrt_eig`, the square-root eigenvalues
  $\sigma\,(4\pi^2|k|^2+\tau^2)^{-\alpha/2}$ for every wavenumber $k=(k_x,k_y)$, and zeros the mean mode.
- `sample(n)` draws complex white noise, scales it by `sqrt_eig`, and takes the real part of the inverse FFT.

In [ ]:
class GaussianRF2d:
    """Sampler for a mean-zero Gaussian random field on the 2D periodic torus [0,1)^2.

    The covariance operator is C = sigma^2 * (-Delta + tau^2 I)^(-alpha), whose
    eigenvalues in the Fourier basis are
        sigma^2 * (4*pi^2*|k|^2 + tau^2)^(-alpha),
    for wavenumber k = (k_x, k_y). Larger alpha / tau produce smoother fields.
    """

    def __init__(self, size, alpha=2.5, tau=7.0, sigma=None, device=None):
        self.size = size
        self.device = device
        d = 2
        if sigma is None:
            sigma = tau ** (0.5 * (2 * alpha - d))

        k_max = size // 2
        wavenumbers = torch.cat(
            (torch.arange(0, k_max, device=device), torch.arange(-k_max, 0, device=device)), 0
        ).repeat(size, 1)
        k_x = wavenumbers.transpose(0, 1)
        k_y = wavenumbers

        eig = (size ** 2) * math.sqrt(2.0) * sigma * (
            (4 * (math.pi ** 2) * (k_x ** 2 + k_y ** 2) + tau ** 2) ** (-alpha / 2.0)
        )
        eig[0, 0] = 0.0  # zero out the mean (DC) component
        self.sqrt_eig = eig

    def sample(self, n_samples):
        """Returns n_samples fields of shape (n_samples, size, size), real-valued."""
        noise = torch.randn(n_samples, self.size, self.size, dtype=torch.cfloat, device=self.device)
        coeffs = self.sqrt_eig * noise
        field = torch.fft.ifft2(coeffs).real
        return field

## Try it

Draw four fields, plot them, and see how `alpha` controls smoothness.

In [ ]:
import matplotlib.pyplot as plt

torch.manual_seed(0)
grf = GaussianRF2d(size=64, alpha=2.5, tau=7.0)
fields = grf.sample(4)
print("samples:", tuple(fields.shape), "| mean ~", fields.mean().item(), "| std ~", fields.std().item())

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, f in zip(axes, fields):
    ax.imshow(f, cmap="RdBu_r"); ax.axis("off")
plt.suptitle("Four initial vorticity fields (alpha=2.5, tau=7)"); plt.show()

In [ ]:
# Rougher vs. smoother fields
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
for ax, alpha in zip(axes, (1.5, 2.5, 4.0)):
    torch.manual_seed(1)
    ax.imshow(GaussianRF2d(64, alpha=alpha, tau=7.0).sample(1)[0], cmap="RdBu_r")
    ax.set_title(f"alpha = {alpha}"); ax.axis("off")
plt.show()